(파이토치 트랜스포머를 활용한) 자연어 처리와 컴퓨터비전 심층학습

p.286~p.298 Word2Vec

### 모델실습: Skip-gram

```
# 임베딩 클래스
embedding = torch.nn.Embedding(
    num_embeddings,
    embedding_dim,
    padding_idx = None,
    norm_type=2.0
)
```


In [1]:
# 6.3 기본 skip_gram 클래스
from torch import nn

class VanillaSkipgram(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings = vocab_size,
        embedding_dim = embedding_dim
    )
    self.linear = nn.Linear(
        in_features = embedding_dim,
        out_features = vocab_size
    )
  def forward(self, input_ids):
    embeddings = self.embedding(input_ids)
    output = self.linear(embeddings)
    return output

In [3]:
!pip install Korpora

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 5.8 MB/s eta 0:00:00


In [2]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 35.9 MB/s eta 0:00:00


In [3]:
# 6.4 영화 리뷰 데이터세트 전처리
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 132MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 62.1MB/s]


[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [4]:
# 6.5 단어 사전 구축
from collections import Counter


def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab


vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [5]:
# 6.6 Skip-gram의 단어 쌍 추출
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs


word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [6]:
# 6.7 인덱스 쌍 변환
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs


index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [7]:
# 6.8 데이터로더 적용
import torch
from torch.utils.data import TensorDataset, DataLoader


index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [8]:
# 6.9 Skip-gram 모델 준비 작업
from torch import optim


device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

In [9]:
# 6.10 모델 학습
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(dataloader)
    print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

Epoch :    1, Cost : 6.200
Epoch :    2, Cost : 5.982
Epoch :    3, Cost : 5.933
Epoch :    4, Cost : 5.903
Epoch :    5, Cost : 5.880
Epoch :    6, Cost : 5.862
Epoch :    7, Cost : 5.847
Epoch :    8, Cost : 5.834
Epoch :    9, Cost : 5.823
Epoch :   10, Cost : 5.812


In [10]:
# 6.11 임베딩 값 추출
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[-0.8698445  -0.8519977   0.30704814  0.8658896   0.35173047  0.04019952
  0.9902187  -0.01538251  0.3131457   0.18748859  0.68524396 -0.55576223
  0.3212968   0.24484754  0.43492794  0.18049416  0.5979581  -0.74520236
  2.0229433  -0.00571885  1.5320908   0.8614226  -1.0352459   1.3620625
  1.5376545   1.7874254   0.44490126 -0.5711787  -0.14952815  0.55901617
 -0.9023608   0.2513891   0.43959275  0.30115822 -0.00627682 -1.3952174
  1.9948277   0.14139529  1.9234339   1.0730745  -0.30980188  0.8829765
 -0.42414275  0.45183805 -0.5966054  -0.78918725 -0.4175743  -0.03007896
 -0.55919737 -1.0241133   0.8917407   0.02656409  0.01590282 -1.827741
  0.6622158   0.3753651  -0.35646632  0.04242388  0.30710387 -0.43312204
 -0.9212982  -1.023888    1.2409922  -0.91633505 -1.247379   -1.2985712
 -0.093159    0.7996504   0.58321357  0.11551011  0.05897852 -0.5657419
  0.7980947   0.56547755 -0.56818736 -0.32744774 -0.6584269  -0.4750575
  0.48963282  1.766804    0.5361287  -1.4934485  -0.8163

In [11]:
# 6.12 단어 임베딩 유사도 계산
import numpy as np
from numpy.linalg import norm


def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1 : n + 1]
    return top_n


cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5 개 단어
말기 - 유사도 : 0.3135
배우 - 유사도 : 0.2938
재미있고 - 유사도 : 0.2684
웃다가 - 유사도 : 0.2620
많네 - 유사도 : 0.2523


### 모델 실습: Gensim
```
word2vec = gensim.models.Word2Vec(
  sentences = None,
  corpus_file = None,
  vector_size = 100,
  alpha=0.025,
  window=5,
  min_count = 5,
  workers = 3,
  sg = 0,
  hs = 0,
  cbow_mean = 1,
  negative = 5,
  ns_exponent = 0.75,
  max_final_vocab = None,
  epochs = 5,
  batch_words = 10000
)
```

In [12]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 34.3 MB/s eta 0:00:00


In [13]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt


corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [14]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]

In [15]:
# 6.13 Word2Vec 모델 학습
from gensim.models import Word2Vec


word2vec = Word2Vec(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    epochs=3,
    max_final_vocab=10000
)

In [16]:
# 6.14 임베딩 추출 및 유사도 계산
word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2="연기력"))

[ 0.10897412 -0.23304237  0.35880095  0.6096753   0.04085878  0.05393863
  0.30527708 -0.07802825 -0.53963786  0.35368583 -0.07274751 -0.36408737
 -0.28478557 -0.0027758  -0.24995872 -0.21081267 -0.24184935  0.42718956
 -0.32451975  0.4298279   0.57129925  0.51111597 -0.11536384 -0.12009229
 -0.01335851 -0.01829759 -0.49896446  0.04612665  0.06278162 -0.11896846
 -0.44002634  0.27828607  0.19464912 -0.04380757 -0.18777883 -0.4098006
  0.13364868 -0.34310782 -0.0071446  -0.45550764 -0.1811514   0.23923914
 -0.04858228 -0.31346872 -0.15926768  0.15989864 -0.41267326 -0.2869429
  0.09754483 -0.06514003  0.887314    0.37347126  0.02935237  0.30614483
 -0.52152413 -0.49214864  0.14491881  0.09603279 -0.32957107  0.06827597
 -0.07542683 -0.06946963  0.1077019   0.03991868 -0.28864828  0.01024274
  0.01477355  0.227579    0.22131261 -0.13876496 -0.22306879 -0.25873172
 -0.47237226  0.15977107 -0.12434993  0.03400101 -0.21274993 -0.45017827
 -0.08376799  0.38966995  0.01491745 -0.07638395  0.2